In [1]:
import pickle 
import os
from pathlib import Path
from scipy import stats
import numpy as np
import datetime
import numpy as np
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import yfinance as yf


### Modelling Phase 2

| Commodity | ARMA (p, q) | GJR-GARCH (p, o, q) |
| :--- | :--- | :--- |
| **NG** | 1, 0 | 1, 0, 1 |
| **Copper** | 0, 0 | 1, 1, 1 |
| **Brent** | 0, 0 | 1, 1, 1 |
| **Corn** | 0, 0 | 1, 1, 1 |
| **WTI** | 3, 3 | 1, 1, 1 |
| **Cattle** | 0, 0 | 1, 0, 1 |
| **Cocoa** | 0, 0 | 1, 0, 1 |

*Note: The `o` parameter represents the asymmetry indicator.*

In [2]:
%pwd

'c:\\Users\\BudgetGamer\\Projects\\Ian\\CommodityProject\\SystemicRiskCopulaProject\\Modelling'

In [3]:
current_path = Path.cwd()
try:
    project_root = next(p for p in current_path.parents if p.name == "CommodityProject")
except StopIteration:
    print(Path.cwd())
except e as exception:
    print(e)

In [4]:
%cd $project_root   

c:\Users\BudgetGamer\Projects\Ian\CommodityProject


In [5]:
data_dir = os.path.join(project_root, "SystemicRiskCopulaProject", "Data")

In [6]:
script_dir = os.path.join(project_root, "SystemicRiskCopulaProject", "Scripts")

### Reading data

In [7]:
%cd $data_dir

c:\Users\BudgetGamer\Projects\Ian\CommodityProject\SystemicRiskCopulaProject\Data


In [8]:
file_list = os.listdir(data_dir)

for file in file_list:
    if file.endswith(".pkl"):
        _file = Path(file).stem
        with open(os.path.join(data_dir, file), "rb") as f:
            globals()[_file] = pickle.load(f)
        print(f"Loaded {file} with keys: {list(globals()[_file].keys())}")
        

Loaded arch_dict.pkl with keys: ['NG', 'Copper', 'Brent', 'Corn', 'WTI', 'Cattle', 'Cocoa']
Loaded arma_dict.pkl with keys: ['NG', 'Copper', 'Brent', 'Corn', 'WTI', 'Cattle', 'Cocoa']
Loaded cleaned_dfs.pkl with keys: ['NG', 'Copper', 'Brent', 'Corn', 'WTI', 'Cattle', 'Cocoa']
Loaded dfs.pkl with keys: ['NG', 'Copper', 'Brent', 'Corn', 'WTI', 'Cattle', 'Cocoa']


### Starting Copula maths

Now we have the conditional mean and conditional variance, and also the standardized residuals, $\eta$, we can start the Copula math. ARMA will be able to generate the mean and standard residual along iwth condiotional variance would be able to generate variance involved. The only source of randomness or stochastic nature is shock term, $\eta$. <br> <br>

$$R_{i,t} = \mu_{i,t} + \sigma_{i,t} \cdot \eta_{i,t} 




$\mu_{i,t}$ :  deterministic - based on yesterdays return, R_{i,t-1} <br>
$\sigma$ : deterministic - based on yesterdays variance <br>
$\eta$ : Standardized shock - Random component <br>

### How the Co-Simulation Actually Unfolds (Step-by-Step)

Suppose you are simulating tomorrow ($t = T+1$) for $C_1$ (Brent) and $C_2$ (WTI).

#### Step A: Determine the deterministic environment

    Before tomorrow's trading bell rings, you already know today's closing prices.
        * You calculate $\mu_{1, T+1}$ and $\mu_{2, T+1}$ using their respective ARMA equations.
        * You calculate $\sigma_{1, T+1}$ and $\sigma_{2, T+1}$ using their respective GARCH equations.
                (Notice: these numbers are fixed. No random number generator is called here.)
#### Step B: Call the Copula for the Joint Shock
    To see what unexpected shock hits the market tomorrow, you sample from the fitted multivariate Copula:
        * The Copula draws correlated uniform numbers:$$(u_1, u_2)  ~ \text{Copula}$$
        
            (If Brent plunges in the copula draw, $u_1 = 0.01$; because of tail dependence, $u_2$ is automatically drawn as $0.012$.)
        * Invert each $u_i$ back through its individual Skewed-$t$ CDF (the inverse Probability Integral Transform, $F_i^{-1}$):
        
            $$\eta_1 = F_{\text{Skew-}t, 1}^{-1}(u_1) = -3.8\sigma$$
            $$\eta_2 = F_{\text{Skew-}t, 2}^{-1}(u_2) = -3.5\sigma$$
                                (-3.8 = F_-1(0.01) and -3.5 = F-1) Those numbers (-3.8 and -3.5) were hypothetical examples I used to illustrate what a "joint crash" looks like mathematically.
                We are transforming CDF(0,1) back to real world shock here. 
                The shocks $(\eta_1, \eta_2)$ now contain all the non-linear tail dependence, joint crash correlation, and cross-asset linkage.

#### Step C: Synthesize Tomorrow's Return
    Now, scale and shift the shocks using the pre-computed deterministic volatility and mean:
    
    $$R_{1, T+1} = \mu_{1, T+1} + \sigma_{1, T+1} \cdot \eta_1$$
    $$R_{2, T+1} = \mu_{2, T+1} + \sigma_{2, T+1} \cdot \eta_2$$
    
    Because $\eta_1$ and $\eta_2$ plummeted together in the simulation, $R_1$ and $R_2$ plunge together.
    
#### What Happens at Step $t = T+2$ (Dynamic Feedback)?

    This is where the magic happens over multi-day simulations.
    When you advance to the next day ($T+2$):
        
        $C_1$'s new conditional volatility $\sigma_{1, T+2}$ is calculated using $\epsilon_{1, T+1} = \sigma_{1, T+1} \cdot \eta_1$.
        
        $C_2$'s new conditional volatility $\sigma_{2, T+2}$ is calculated using $\epsilon_{2, T+1} = \sigma_{2, T+1} \cdot \eta_2$.
        
    Because both assets took a joint crash in step $T+1$ through the copula shocks ($\eta_1, \eta_2$), both assets' GARCH models automatically spike their volatility forecasts simultaneously for step $T+2$.

### Coding

#### Data for checking

Train data: 2016 to 2025 <br>
Test data: 2026 till today <br>
Note: I forgot to split the data into test and train :(

In [9]:
#Henry Hub Natural Gas
df_ng = yf.download("NG=F", start="2026-01-01", end="2026-09-31", interval="1d")
#Copper
df_cu = yf.download("HG=F", start="2026-01-01", end="2026-09-31", interval="1d")
#Brent
df_bz = yf.download("BZ=F", start="2026-01-01", end="2026-09-31", interval="1d")
#Corn
df_zc = yf.download("ZC=F", start="2026-01-01", end="2026-09-31", interval="1d")
#Crude Oil WTI
df_cl = yf.download("CL=F", start="2026-01-01", end="2026-09-31", interval="1d")
#Cattle-Live
df_le = yf.download("LE=F", start="2026-01-01", end="2026-09-31", interval="1d")
#Cocoa
df_cc = yf.download("CC=F", start="2026-01-01", end="2026-09-31", interval="1d")

[*********************100%***********************]  1 of 1 completed

1 Failed download:
['NG=F']: ValueError('day is out of range for month')
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['HG=F']: ValueError('day is out of range for month')
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['BZ=F']: ValueError('day is out of range for month')
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['ZC=F']: ValueError('day is out of range for month')
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['CL=F']: ValueError('day is out of range for month')
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['LE=F']: ValueError('day is out of range for month')
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['CC=F']: ValueError('day is out of range for month')

####  model

In [10]:
arma_dict.keys()

dict_keys(['NG', 'Copper', 'Brent', 'Corn', 'WTI', 'Cattle', 'Cocoa'])

In [11]:
crude = arch_dict.get('Brent')

In [12]:
crude['best_resid'].params

omega       0.000014
alpha[1]    0.078708
gamma[1]    0.033032
beta[1]     0.880992
eta         5.439791
lambda     -0.167054
Name: params, dtype: float64

In [13]:
crude['cond_vol']

2       0.043516
3       0.046009
4       0.043616
7       0.041164
8       0.043974
          ...   
3885    0.027281
3886    0.027108
3887    0.028998
3888    0.027630
3889    0.026834
Name: cond_vol, Length: 2657, dtype: float64

In [14]:
dfs

{'NG':            Date  Close    Volume  Percentage_Returns  Relative_change  \
 2    2016-01-06  2.267  166278.0           -0.024946           -0.115   
 3    2016-01-07  2.382  214462.0            0.050728           -0.090   
 4    2016-01-08  2.472  163907.0            0.037783            0.000   
 5    2016-01-09  2.472  163907.0            0.000000            0.000   
 6    2016-01-10  2.472  163907.0            0.000000            0.076   
 ...         ...    ...       ...                 ...              ...   
 3885 2026-08-24  2.782   90855.0            0.003246            0.012   
 3886 2026-08-25  2.770   42228.0           -0.004313           -0.072   
 3887 2026-08-26  2.842   68681.0            0.025993           -0.065   
 3888 2026-08-27  2.907  188571.0            0.022871            0.019   
 3889 2026-08-28  2.888  132710.0           -0.006536              NaN   
 
       Log_Returns  
 2       -0.025263  
 3        0.049483  
 4        0.037087  
 5        0.000000  

In [15]:
import importlib

In [16]:
%cd $script_dir 

c:\Users\BudgetGamer\Projects\Ian\CommodityProject\SystemicRiskCopulaProject\Scripts


In [17]:
import BasicOperations

In [18]:
for _name, _df in dfs.items():
    print(f' {_name} has {len(dfs[_name])} rows before removing holidays.')
    dfs[_name] = BasicOperations.removeHolidays(_df)
    print(f' {_name} has {len(dfs[_name])} rows after removing holidays.')

 NG has 3888 rows before removing holidays.
 NG has 2657 rows after removing holidays.
 Copper has 3888 rows before removing holidays.
 Copper has 2657 rows after removing holidays.
 Brent has 3888 rows before removing holidays.
 Brent has 2657 rows after removing holidays.
 Corn has 3888 rows before removing holidays.
 Corn has 2657 rows after removing holidays.
 WTI has 3888 rows before removing holidays.
 WTI has 2657 rows after removing holidays.
 Cattle has 3888 rows before removing holidays.
 Cattle has 2657 rows after removing holidays.
 Cocoa has 3888 rows before removing holidays.
 Cocoa has 2657 rows after removing holidays.


In [19]:
trimmedDates = dfs['NG']['Date']

In [23]:
trimmedDates[(trimmedDates.dt.year == 2025) & (trimmedDates.dt.month == 12)]

3619   2025-12-01
3620   2025-12-02
3621   2025-12-03
3622   2025-12-04
3623   2025-12-05
3626   2025-12-08
3627   2025-12-09
3628   2025-12-10
3629   2025-12-11
3630   2025-12-12
3633   2025-12-15
3634   2025-12-16
3635   2025-12-17
3636   2025-12-18
3637   2025-12-19
3640   2025-12-22
3641   2025-12-23
3642   2025-12-24
3644   2025-12-26
3647   2025-12-29
3648   2025-12-30
3649   2025-12-31
Name: Date, dtype: datetime64[s]